# Similar Category-Category Mapping with LLM Validation

Handles both Category and OPC - category to category mapping in this script.

In [ ]:
import subprocess
subprocess.run(["git", "clone", "<your-repo-url>", "/home/jupyter/ds-search-relevancy-repo/catg-catg-kw-catg"], check=True)

### Install packages and dependencies

In [2]:
!pip install -r /home/jupyter/ds-search-relevancy-repo/catg-catg-kw-catg/requirements.txt

Looking in indexes: https://pypi.org/simple, https://asia-south1-python.pkg.dev/prj-onlinesales-prod-01/os-python-packages/simple/
  Obtaining dependency information for boto from https://files.pythonhosted.org/packages/23/10/c0b78c27298029e4454a472a1919bde20cb182dab1662cec7f2ca1dcc523/boto-2.49.0-py2.py3-none-any.whl.metadata
  Using cached annoy-1.17.3.tar.gz (647 kB)
  Preparing metadata (setup.py) ... done
  Obtaining dependency information for openpyxl from https://files.pythonhosted.org/packages/58/d9/796181a30827b12101786c21301f0f4536597a9249530916b1fdb5bbad91/openpyxl-3.1.3-py2.py3-none-any.whl.metadata
  Using cached openpyxl-3.1.3-py2.py3-none-any.whl.metadata (2.5 kB)
  Obtaining dependency information for tabulate from https://files.pythonhosted.org/packages/40/44/4a5f08c96eb108af5cb50b41f76142f0afa346dfa99d5296fe7202a11854/tabulate-0.9.0-py3-none-any.whl.metadata
  Using cached tabulate-0.9.0-py3-none-any.whl.metadata (34 kB)
  Obtaining dependency information for humanfri

In [ ]:
import boto3
import botocore
import annoy 
import openpyxl
import tabulate
import humanfriendly
import nltk
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
from google import genai
from google.genai import types
from google.cloud import bigquery, secretmanager 
import argparse 
import io 
import re 
import time 
import datetime 
import tempfile
import json

from typing import Optional, Any, List, Tuple
import string
from functools import lru_cache

import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")
STOPWORDS = stopwords.words("english")

from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
from google.cloud import bigquery
from google import genai
from google.genai import types

import os, sys

# Import modules
_REPO_ROOT = "/home/jupyter/ds-search-relevancy-repo/catg-catg-kw-catg"
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)
    

# Setup connection with S3
from utils.bq_setup import bq_query
from utils.secrets import fetch_secret
from utils.sql_loader import load_sql
from utils.text_utils import preprocess
from utils.validation_cache import load_cache, save_cache, _dedupe_cache, add_manual_entry
from utils.prompts import SYSTEM_INSTR as SYSTEM_INSTR_TEMPLATE, SYSTEM_INSTR_2 as SYSTEM_INSTR_2_TEMPLATE

import gzip
from tempfile import NamedTemporaryFile
from tempfile import mkdtemp as TemporaryDirectory
from boto.s3.connection import Bucket, Key, S3Connection
from osUtilsV2.log_utils import LogUtils
from osUtilsV2.data_utils import DataUtils
from osClient4pyV2.s3_client import S3Client

# Load pipeline settings
from config import (
    USE_MODULE_URL, USE_MODEL_GCS_PATH,
    MODEL_CASCADE, DEFAULT_MODEL_ID, MODEL_PRICING, MAX_CANDIDATES_PER_BATCH,
    MAX_THRESHOLD, THRESHOLD_STEP,
    CLIENT_PIPELINE_TYPE, PIPELINE_PARAMS,
    S3_CATG_MODEL_OUTPUT, S3_CATG_VALIDATION_CACHE,
    S3_OPC_MODEL_OUTPUT, S3_OPC_VALIDATION_CACHE,
)

tf.config.set_visible_devices([], 'GPU')

# Points to SQL queries
_INPUTS_DIR = os.path.join(_REPO_ROOT, "inputs")

### Universal Sentence Encoder

In [ ]:
def load_use_model():
    try:
        print(f"Loading USE model from GCS: {USE_MODEL_GCS_PATH}")
        return hub.load(USE_MODEL_GCS_PATH)
    except Exception:
        print(f"GCS cache not found. Downloading from TF Hub: {USE_MODULE_URL}")
        loaded_model = hub.load(USE_MODULE_URL)
        print(f"Saving USE model to GCS: {USE_MODEL_GCS_PATH}")
        tf.saved_model.save(loaded_model, USE_MODEL_GCS_PATH)
        print(f"USE model cached to GCS.")
        return loaded_model

model = load_use_model()

### Product Title Processing

In [ ]:
def prod_desc_manipulation(
    sku_desc_df: pd.DataFrame,
    chunk_size: int,
    leaf_catg_cnt: int,
    top_skus_cnt: int,
    catg_col: List[str],
    embedding_order: str = "name_first",
) -> Tuple[pd.DataFrame, List[str]]:

    # Removing NaN's from the category columns and filling it with '' and removing unwanted characters
    sku_desc_df[catg_col] = sku_desc_df[catg_col].fillna('').apply(lambda col: col.str.lower())
    sku_desc_df["not_null_catg"] = sku_desc_df[catg_col].apply(
        lambda row: [col for col in row if col != ''], axis=1
    )
    sku_desc_df['catg_concat'] = sku_desc_df["not_null_catg"].apply(lambda row: ">".join(row))
    sku_desc_df['leaf_catg_concat'] = sku_desc_df['not_null_catg'].apply(
        lambda row: ">".join(row[-leaf_catg_cnt:])
    )

    # Defining if the marketplace is catg or opc and further concating with the e_name/leaf_catg_concat making it raw input to the model
    if embedding_order == "leaf_first":
        sku_desc_df['desc_catg_concat'] = sku_desc_df['leaf_catg_concat'] + ">" + sku_desc_df['e_name']
    else:
        sku_desc_df['desc_catg_concat'] = sku_desc_df['e_name'] + ">" + sku_desc_df['leaf_catg_concat']
    print("Product desc dataset initial shape: ", sku_desc_df.shape)

    # Removing the cases with no category mappping, if any
    sku_desc_df_v2 = sku_desc_df.loc[~(sku_desc_df['catg_concat'] == '')]
    print("After removing records without categories", sku_desc_df_v2.shape)

    # Ranking the SKUs based on sok_sales
    sku_desc_df_v2["category_rank"] = sku_desc_df_v2.groupby("catg_concat")["sok_sales_usd"].rank(
        ascending=False, method="first"
    )

    # Filtering the best performing SKUs of each category combination
    sku_desc_df_v4 = sku_desc_df_v2.loc[sku_desc_df_v2["category_rank"] <= top_skus_cnt].reset_index(drop=True)

    # Preprocessing the model input column
    tqdm.pandas(desc="Preprocessing the description column for the top skus")
    sku_desc_df_v4['desc_catg_concat_processed'] = sku_desc_df_v4['desc_catg_concat'].progress_apply(preprocess)
    descriptions_list = sku_desc_df_v4['desc_catg_concat_processed'].tolist()
    return sku_desc_df_v4, descriptions_list

### Creating word embeddings for each description

In [ ]:
def create_word_embeddings(descriptions_list: List[str], chunk_size: int):
    # Create an empty list to store the vectors
    category_desc_list_vectors = []

    # Iterate over the descriptions_list in chunks
    for i in tqdm(range(0, len(descriptions_list), chunk_size), desc="Calculating vectors for category description list",):
        # Get the current chunk
        chunk = descriptions_list[i:i + chunk_size]
        # Calculate vectors for the current chunk
        vectors = model(chunk)
        # Append the vectors to the list
        category_desc_list_vectors.extend(vectors)

    # Convert the list of vectors to a numpy array
    return np.array(category_desc_list_vectors)



### Category Embedding Aggregation

In [ ]:
def create_catg_embedding(category_desc_list_vectors, df: pd.DataFrame) -> pd.DataFrame:
    # Calculate the average pooling for each category
    category_embeddings = []
    categories = []

    # Get unique categories in the DataFrame
    unique_categories = df['catg_concat'].unique()

    for category in tqdm(unique_categories, desc="Calculating category embeddings"):
        category_df = df[df['catg_concat'] == category]
        valid_indices = [i for i in category_df.index if i < len(category_desc_list_vectors)]

        if valid_indices:
            category_desc_embeddings = [category_desc_list_vectors[i] for i in valid_indices]
            category_embedding = np.mean(category_desc_embeddings, axis=0)
        else:
            # If there are no valid indices, set the category embedding to NaN or any default value
            category_embedding = np.nan

        category_embeddings.append(category_embedding)
        categories.append(category)

    category_embeddings_df = pd.DataFrame({
        'catg_concat': categories,
        'category_embedding': category_embeddings
    })

    return category_embeddings_df

### ANN Index Construction

In [ ]:
def build_annoy_index(n_trees, df: pd.DataFrame):
    # Build Annoy Index for other search terms
    index = annoy.AnnoyIndex(df['category_embedding'][0].shape[0], 'angular')
    for i, vector in tqdm(enumerate(df['category_embedding']), desc="Building index"):
        if not isinstance(vector, float) and not np.isnan(vector).any():
            index.add_item(i, vector.flatten())
        else:
            print(f"Skipping category {df['catg_concat'][i]} because its embedding is invalid.")
    index.build(n_trees)
    return index

### Category-Category Similarity

In [ ]:
def calculate_catg_catg_similarity(n_neighbors, index, df: pd.DataFrame) -> pd.DataFrame:
    # Find similar search terms for each target search term
    output_df_list = []

    for i in tqdm(range(len(df)), desc="Finding similar categories"):
        category_vector = df["category_embedding"][i]
        similar_category_indices = index.get_nns_by_vector(category_vector, n_neighbors)
        similar_category = [df["catg_concat"][j] for j in similar_category_indices if j < len(df)]
        similarities = [
            cosine_similarity([category_vector], [df["category_embedding"][j]])[0][0]
            for j in similar_category_indices
            if j < len(df)
        ]
        output_df = pd.DataFrame({
            "category_concat": [df["catg_concat"][i]] * len(similar_category),
            "similar_category": similar_category,
            "similarity_score": similarities,
        })
        output_df_list.append(output_df)

    # Concat the list
    output_df = pd.concat(output_df_list)
    return output_df

### Advertisable Category Processing

In [ ]:
def adv_prod_desc_manipulation(adv_sku_desc_df: pd.DataFrame, chunk_size: int, leaf_catg_cnt: int, catg_col: List[str],) -> pd.DataFrame:
    
    # Removing NaN's from the category columns and filling it with '' and removing unwanted characters
    adv_sku_desc_df[catg_col] = adv_sku_desc_df[catg_col].fillna('').apply(lambda col: col.str.lower())
    adv_sku_desc_df["not_null_catg"] = adv_sku_desc_df[catg_col].apply( lambda row: [col for col in row if col != ''], axis=1)
    adv_sku_desc_df['catg_concat'] = adv_sku_desc_df["not_null_catg"].apply(lambda row: ">".join(row))
    adv_sku_desc_df['leaf_catg_concat'] = adv_sku_desc_df['not_null_catg'].apply( lambda row: ">".join(row[-leaf_catg_cnt:]))

    # Removing the cases with no category mappping, if any
    adv_sku_desc_df_v2 = adv_sku_desc_df.loc[~(adv_sku_desc_df['catg_concat'] == '')]
    
    adv_catg = adv_sku_desc_df_v2[['catg_concat']].drop_duplicates()
    return adv_catg



### Output Filtering

In [ ]:
def output_manipulations(min_score_threshold: float, df: pd.DataFrame) -> pd.DataFrame:
    # Applying a threhold on the output
    output_df_v2 = df.loc[df['similarity_score'] >= min_score_threshold]
    
    print("Output shape after implementing the score threshold: ", output_df_v2.shape)
    
    # Removing the 'nan' category in case present
    output_df_v3 = output_df_v2[~(output_df_v2['category_concat'] == "")]
    
    return output_df_v3

### LLM helpers

In [ ]:
def _build_category_input(source_catg, candidates_df):
    # Getting the pairings from the embeddings & only sending source category and its similar category with row number to ensure data integrity
    lines = [f"Source category: {source_catg}", ""]
    lines.append("row_number\tsimilar_category")
    for idx, (_, row) in enumerate(candidates_df.iterrows(), start=1):
        lines.append(f"{idx}\t{row['similar_category']}")
    return "\n".join(lines)

In [ ]:
def _parse_llm_response(resp_text, batch_size):
    # Parsing LLM response & safety handling
    rows = []
    for line in resp_text.strip().split('\n'):
        if not line.strip():
            continue
        # Split by tab
        parts = line.split('\t')
        row_number = None
        llm_flag = None
        reason_parts = []

        for part in parts:
            ps = part.strip()
            pu = ps.upper()
            if pu in ('RELEVANT', 'LOOSELY_RELEVANT', 'IRRELEVANT') and llm_flag is None:
                llm_flag = pu
                continue
            # Finding row number
            if row_number is None:
                try:
                    row_number = int(ps)
                    continue
                except ValueError:
                    pass
            if ps:
                reason_parts.append(ps)

        if row_number is None:
            continue
        if row_number < 1 or row_number > batch_size:
            continue

        # Finding LLM flag
        if llm_flag is None:
            llm_flag = 'IRRELEVANT'
        # Anything left - llm_reason
        llm_reason = ' - '.join(reason_parts) if reason_parts else llm_flag

        rows.append({'row_number': row_number, 'llm_reason': llm_reason, 'llm_flag': llm_flag})

    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=['row_number', 'llm_reason', 'llm_flag'])

In [ ]:
def _send_llm_request(client, model_id, batch_text, system_instr, max_output_tokens=2000):
    config = types.GenerateContentConfig(
        temperature=0,
        system_instruction=system_instr,
        max_output_tokens=max_output_tokens,
        seed=42,
    )
    return client.models.generate_content(model=model_id, contents=batch_text, config=config)

In [ ]:
def _save_output_files(final_df, out_path, client_id, cache_df):
    validated_df = final_df[['category_concat', 'similar_category', 'similarity_score', 'llm_reason', 'llm_flag']].copy()
    validated_df.insert(0, 'row_number', range(1, len(validated_df) + 1))
    validated_df.to_csv(out_path, sep='\t', index=False)
    print(f"  Validated output saved to: {out_path}")

    llm_out_path = out_path.replace("_validated.tsv", "_llm_output.tsv")
    llm_only_df = final_df[['llm_reason', 'llm_flag']].copy()
    llm_only_df.insert(0, 'row_number', range(1, len(llm_only_df) + 1))
    llm_only_df.to_csv(llm_out_path, sep='\t', index=False)

In [ ]:
def _send_category_batch(source_catg, candidates_df, client, model_id, system_instr, pricing):
    batch_text = _build_category_input(source_catg, candidates_df)
    current_batch_size = len(candidates_df)
    batch_row_ids = candidates_df['row_id'].tolist()

    response = _send_llm_request(client, model_id, batch_text, system_instr=system_instr)

    usage = response.usage_metadata
    input_tokens = usage.prompt_token_count or 0
    output_tokens = usage.candidates_token_count or 0
    cost = (input_tokens / 1_000_000) * pricing["input"] + (output_tokens / 1_000_000) * pricing["output"]

    resp_df = _parse_llm_response(response.text, current_batch_size)
    if resp_df.empty:
        return None, input_tokens, output_tokens, cost

    resp_df['row_id'] = resp_df['row_number'].apply(
        lambda rn: batch_row_ids[rn - 1] if 1 <= rn <= len(batch_row_ids) else None
    )
    resp_df = resp_df.dropna(subset=['row_id'])
    resp_df['row_id'] = resp_df['row_id'].astype(int)
    resp_df = resp_df[['row_id', 'llm_reason', 'llm_flag']]

    return resp_df, input_tokens, output_tokens, cost

In [ ]:
def _run_llm_pass(df_to_process, client, model_id, system_instr, pricing, max_candidates_per_batch=25, pass_label="Pass 1"):
    all_validations = []
    cumulative_input_tokens = 0
    cumulative_output_tokens = 0
    cumulative_cost = 0.0

    grouped = df_to_process.groupby('category_concat', sort=False)
    total_groups = len(grouped)
    batch_num = 0

    for source_catg, group_df in tqdm(grouped, desc=f"{pass_label} LLM Processing", total=total_groups):
        for sub_start in range(0, len(group_df), max_candidates_per_batch):
            sub_batch = group_df.iloc[sub_start:sub_start + max_candidates_per_batch]
            batch_num += 1

            try:
                resp_df, b_in, b_out, b_cost = _send_category_batch(
                    source_catg, sub_batch, client, model_id, system_instr, pricing
                )
            except KeyboardInterrupt:
                print(f"\n  Interrupted at batch {batch_num}.")
                break
            except Exception as e:
                print(f"  Error at batch {batch_num} ({source_catg}): {e}")
                continue

            cumulative_input_tokens += b_in
            cumulative_output_tokens += b_out
            cumulative_cost += b_cost

            if batch_num % 10 == 0 or batch_num == total_groups:
                print(f"  {pass_label} Batch {batch_num}: Cumulative cost so far: ${cumulative_cost:.6f}")

            if resp_df is not None:
                all_validations.append(resp_df)

    if not all_validations:
        return df_to_process, cumulative_input_tokens, cumulative_output_tokens, cumulative_cost

    validations_df = pd.concat(all_validations, ignore_index=True).drop_duplicates(subset=['row_id'])
    df_merged = df_to_process.merge(validations_df, on='row_id', how='left')

    max_retries = 3
    for retry_num in range(1, max_retries + 1):
        missing_mask = df_merged['llm_flag'].isna() if 'llm_flag' in df_merged.columns else pd.Series([True] * len(df_merged))
        missing_count = missing_mask.sum()
        if missing_count == 0:
            break

        print(f"  {pass_label} Retry {retry_num}/{max_retries}: {missing_count} rows missing — re-sending to LLM...")
        missing_rows = df_merged[missing_mask].copy()
        retry_validations = []

        for source_catg, group_df in missing_rows.groupby('category_concat', sort=False):
            try:
                resp_df, r_in, r_out, r_cost = _send_category_batch(
                    source_catg, group_df, client, model_id, system_instr, pricing
                )
                cumulative_input_tokens += r_in
                cumulative_output_tokens += r_out
                cumulative_cost += r_cost
                if resp_df is not None:
                    retry_validations.append(resp_df)
            except Exception as e:
                print(f"    {pass_label} Retry failed for {source_catg}: {e}")

        if retry_validations:
            retry_df = pd.concat(retry_validations, ignore_index=True)
            df_merged = df_merged.set_index('row_id')
            df_merged.update(retry_df.set_index('row_id')[['llm_reason', 'llm_flag']])
            df_merged = df_merged.reset_index()
            print(f"    Filled {len(retry_df)}/{missing_count} missing rows")

    return df_merged, cumulative_input_tokens, cumulative_output_tokens, cumulative_cost

### LLM Validation

In [ ]:
def validate_with_llm(input_file_path, client_id, model_id=None, system_instr=None, system_instr_2=None):
    print(f"\n{'='*60}")
    print(f"Two-Pass LLM Validation (Per-Category Batching + Pair Cache)")
    print(f"Input: {input_file_path}")
    print(f"Client: {client_id}")
    print(f"{'='*60}")

    if not os.path.exists(input_file_path):
        print(f"Error: File not found — {input_file_path}")
        return

    df = pd.read_csv(input_file_path, sep='\t')

    if df.empty:
        print("File is empty. Skipping validation.")
        return

    gemini_api_key = fetch_secret(name="gemini_api_key")
    client = genai.Client(api_key=gemini_api_key)
    if model_id is None:
        model_id = DEFAULT_MODEL_ID
    max_candidates_per_batch = MAX_CANDIDATES_PER_BATCH
    pricing = MODEL_PRICING.get(model_id, {"input": 0.10, "output": 0.40})

    df_val = df.reset_index().rename(columns={'index': 'row_id'})
    out_path = input_file_path.replace(".tsv", "_validated.tsv")

    self_match_mask = df_val['category_concat'] == df_val['similar_category']
    df_self = df_val[self_match_mask].copy()
    df_non_self = df_val[~self_match_mask].copy()

    print("\n  Loading pair-level validation cache...")
    cache_df = load_cache(client_id)

    valid_cache = _dedupe_cache(cache_df) if not cache_df.empty else pd.DataFrame(columns=['category_concat', 'similar_category', 'llm_reason', 'llm_flag', 'source'])
    manual_count = (valid_cache['source'] == 'MANUAL').sum() if not valid_cache.empty and 'source' in valid_cache.columns else 0
    if manual_count > 0:
        print(f"  Manual entries preserved: {manual_count}")

    df_with_cache = df_non_self.merge(
        valid_cache[['category_concat', 'similar_category', 'llm_reason', 'llm_flag']],
        on=['category_concat', 'similar_category'],
        how='left',
    )

    cache_hit_mask = df_with_cache['llm_flag'].notna()
    df_cached = df_with_cache[cache_hit_mask].copy()
    df_to_llm = df_with_cache[~cache_hit_mask].drop(columns=['llm_reason','llm_flag']).copy().reset_index(drop=True)

    print(f"\n  Total rows: {len(df_val)}")
    print(f"  Self-matches (removed): {len(df_self)}")
    print(f"  Cache hits (reused): {len(df_cached)}")
    print(f"  Cache misses (sent to LLM): {len(df_to_llm)}")
    print(f"  Unique source categories to process: {df_to_llm['category_concat'].nunique()}")

    if df_to_llm.empty:
        final_df = pd.concat([df_cached], ignore_index=True)
        final_df = final_df.sort_values('row_id').drop(columns=['row_id']).reset_index(drop=True)
        _save_output_files(final_df, out_path, client_id, cache_df)
        print(f"\n  All rows resolved from cache + self-match. Saved to: {out_path}")
        return

    # PASS 1
    print("\n" + "="*40)
    print("PASS 1: Initial Classification")
    print("="*40)

    print("\n[1/6] Processing Pass 1 (category-level batches)...")
    df_llm_merged, p1_input_tokens, p1_output_tokens, p1_cost = _run_llm_pass(
        df_to_llm, client, model_id, system_instr, pricing, max_candidates_per_batch, pass_label="Pass 1"
    )

    pass1_df = pd.concat([df_cached, df_llm_merged], ignore_index=True)
    pass1_df_sorted = pass1_df.sort_values('row_id').drop(columns=['row_id']).reset_index(drop=True)
    pass1_path = out_path.replace("_validated.tsv", "_pass1.tsv")
    pass1_out = pass1_df_sorted[['category_concat', 'similar_category', 'llm_reason', 'llm_flag']].copy()
    pass1_out.insert(0, 'row_number', range(1, len(pass1_out) + 1))
    pass1_out.to_csv(pass1_path, sep='\t', index=False)
    print(f"\n  Pass 1 output saved to: {pass1_path}")

    p1_matched = df_llm_merged['llm_flag'].notna().sum() if 'llm_flag' in df_llm_merged.columns else 0
    p1_relevant = (df_llm_merged['llm_flag'] == 'RELEVANT').sum() if 'llm_flag' in df_llm_merged.columns else 0
    p1_loosely = (df_llm_merged['llm_flag'] == 'LOOSELY_RELEVANT').sum() if 'llm_flag' in df_llm_merged.columns else 0
    p1_irrelevant = (df_llm_merged['llm_flag'] == 'IRRELEVANT').sum() if 'llm_flag' in df_llm_merged.columns else 0
    print(f"  Pass 1 results: {p1_matched} classified — RELEVANT: {p1_relevant}, LOOSELY_RELEVANT: {p1_loosely}, IRRELEVANT: {p1_irrelevant}")

    # PASS 2
    p2_input_tokens = 0
    p2_output_tokens = 0
    p2_cost = 0.0

    loosely_mask = df_llm_merged['llm_flag'] == 'LOOSELY_RELEVANT' if 'llm_flag' in df_llm_merged.columns else pd.Series([False] * len(df_llm_merged))
    df_loosely = df_llm_merged[loosely_mask].copy().reset_index(drop=True)

    if len(df_loosely) > 0:
        print("\n" + "="*40)
        print(f"PASS 2: Re-evaluating {len(df_loosely)} LOOSELY_RELEVANT pairs")
        print("="*40)

        df_loosely_clean = df_loosely.drop(columns=['llm_reason', 'llm_flag']).copy()

        print("\n[3/6] Processing Pass 2 (category-level batches)...")
        df_pass2_merged, p2_input_tokens, p2_output_tokens, p2_cost = _run_llm_pass(
            df_loosely_clean, client, model_id, system_instr_2, pricing, max_candidates_per_batch, pass_label="Pass 2"
        )

        p2_results = df_pass2_merged[df_pass2_merged['llm_flag'].notna()][['row_id', 'llm_reason', 'llm_flag']].copy()
        df_llm_merged = df_llm_merged.set_index('row_id')
        df_llm_merged.update(p2_results.set_index('row_id')[['llm_reason', 'llm_flag']])
        df_llm_merged = df_llm_merged.reset_index()

        p2_relevant = (p2_results['llm_flag'] == 'RELEVANT').sum()
        p2_still_loosely = (p2_results['llm_flag'] == 'LOOSELY_RELEVANT').sum()
        p2_irrelevant = (p2_results['llm_flag'] == 'IRRELEVANT').sum()
        print(f"\n  Pass 2 results: {len(p2_results)} re-classified — RELEVANT: {p2_relevant}, LOOSELY_RELEVANT: {p2_still_loosely}, IRRELEVANT: {p2_irrelevant}")
    else:
        print("\n  No LOOSELY_RELEVANT pairs found — skipping Pass 2.")

    # SAVE FINAL RESULTS
    print("\n[5/6] Saving final results to cache...")

    llm_validated_rows = df_llm_merged[df_llm_merged['llm_flag'].notna()].copy() if 'llm_flag' in df_llm_merged.columns else pd.DataFrame()
    if not llm_validated_rows.empty:
        new_cache_entries = llm_validated_rows[['category_concat', 'similar_category', 'llm_reason', 'llm_flag']].copy()
        cache_df = save_cache(new_cache_entries, cache_df, client_id)

    print("\n[6/6] Saving final output...")
    final_df = pd.concat([df_cached, df_llm_merged], ignore_index=True)
    final_df = final_df.sort_values('row_id').drop(columns=['row_id']).reset_index(drop=True)

    _save_output_files(final_df, out_path, client_id, cache_df)

    total_relevant = (final_df['llm_flag'] == 'RELEVANT').sum()
    total_loosely = (final_df['llm_flag'] == 'LOOSELY_RELEVANT').sum()
    total_irrelevant = (final_df['llm_flag'] == 'IRRELEVANT').sum()
    still_missing = final_df['llm_flag'].isna().sum()

    print(f"\n  Final breakdown:")
    print(f"    RELEVANT: {total_relevant}")
    print(f"    LOOSELY_RELEVANT: {total_loosely}")
    print(f"    IRRELEVANT: {total_irrelevant}")
    print(f"    Self-matches (removed): {len(df_self)}")
    print(f"    Cache hits (reused): {len(df_cached)}")
    if still_missing > 0:
        print(f"    WARNING: {still_missing} rows still unclassified")
    print(f"  Results saved to: {out_path}")

    total_input = p1_input_tokens + p2_input_tokens
    total_output = p1_output_tokens + p2_output_tokens
    total_cost = p1_cost + p2_cost

    print(f"\n--- Token Usage & Cost Summary (Model: {model_id}) ---")
    print(f"  Pass 1 — Input: {p1_input_tokens} | Output: {p1_output_tokens} | Cost: ${p1_cost:.6f}")
    print(f"  Pass 2 — Input: {p2_input_tokens} | Output: {p2_output_tokens} | Cost: ${p2_cost:.6f}")
    print(f"  Total  — Input: {total_input} | Output: {total_output} | Cost: ${total_cost:.6f}")
    print(f"  Pricing — Input: ${pricing['input']}/1M | Output: ${pricing['output']}/1M")
    print(f"  Pair cache: {len(df_cached)} hits")
    print(f"  Self-matches removed: {len(df_self)}")
    print(f"{'='*60}\n")


def _count_batches(df, max_candidates_per_batch=25):
    non_self = df[df['category_concat'] != df['similar_category']]
    return sum(
        max(1, (len(g) + max_candidates_per_batch - 1) // max_candidates_per_batch)
        for _, g in non_self.groupby('category_concat', sort=False)
    )


def resolve_threshold_and_model(raw_output, adv_catg, start_threshold, catg_batch_limit=1000,
                                max_candidates_per_batch=25):
    for model_idx, model_id in enumerate(MODEL_CASCADE):
        threshold = start_threshold

        while True:
            filtered = output_manipulations(threshold, df=raw_output)
            merged = pd.merge(filtered, adv_catg, how='inner',
                              left_on=['similar_category'], right_on=['catg_concat']).drop(columns=['catg_concat'])
            batch_count = _count_batches(merged, max_candidates_per_batch)

            print(f"  [{model_id}] Threshold {threshold:.2f}: {len(merged)} pairs, {batch_count} batches (limit: {catg_batch_limit})")

            if batch_count <= catg_batch_limit:
                print(f"  -> Within limit. Using threshold={threshold:.2f}, model={model_id}")
                return merged, threshold, model_id

            if threshold >= MAX_THRESHOLD:
                break

            threshold = round(threshold + THRESHOLD_STEP, 2)
            if threshold > MAX_THRESHOLD:
                threshold = MAX_THRESHOLD

        if model_idx == len(MODEL_CASCADE) - 1:
            print(f"  -> All models exhausted at max threshold. Using threshold={threshold:.2f}, model={model_id}")
            return merged, threshold, model_id

        print(f"  -> Threshold at max ({MAX_THRESHOLD:.2f}), still {batch_count} batches > {catg_batch_limit}. Trying next model: {MODEL_CASCADE[model_idx + 1]}")

### Run Pipeline

In [ ]:
def main(client_id, run_pipeline=True, run_validation=True):

    # Resolve pipeline type from config (default: catg)
    pipeline_type = CLIENT_PIPELINE_TYPE.get(str(client_id), "catg")
    params = PIPELINE_PARAMS[pipeline_type]
    print(f"Pipeline type: {pipeline_type}")

    if pipeline_type == "opc":
        output_file = f'similar_opc_opc_mapping_{client_id}.tsv'
        s3_output_path = S3_OPC_MODEL_OUTPUT.format(client_id=client_id)
    else:
        output_file = f'similar_catg_catg_mapping_{client_id}.tsv'
        s3_output_path = S3_CATG_MODEL_OUTPUT.format(client_id=client_id)

    full_path = f"/tmp/{output_file}"

    access_details = fetch_secret(name="category_category_similarity", as_json=True)
    access_key = access_details['ACCESS_KEY']
    secret_key = access_details['SECRET_KEY']

    bg_query_str = f"SELECT DISTINCT name FROM reporting.clients WHERE client_id = '{client_id}' LIMIT 1"
    bg_result = bq_query(bg_query_str)
    background = bg_result['name'].iloc[0]
    print(f"Marketplace: {background}")

    system_instr = SYSTEM_INSTR_TEMPLATE.format(background=background)
    system_instr_2 = SYSTEM_INSTR_2_TEMPLATE.format(background=background)

    model_id = DEFAULT_MODEL_ID

    if run_pipeline:
        print(f"Starting full pipeline ({pipeline_type})...")

        query_params = {"client_id": str(client_id)}
        prod_desc_query = load_sql("prod_desc.sql", client_id, params, _INPUTS_DIR)
        adv_prod_desc_query = load_sql("adv_prod_desc.sql", client_id, params, _INPUTS_DIR)

        chunk_size = params["chunk_size"]
        leaf_catg_cnt = params["leaf_catg_cnt"]
        top_skus_cnt = params["top_skus_cnt"]
        catg_cols = params["catg_cols"]
        n_trees = params["n_trees"]
        n_neighbors = params["n_neighbors"]
        min_score_threshold = params["min_score_threshold"]
        catg_batch_limit = params["catg_batch_limit"]
        embedding_order = params["embedding_order"]

        model_input = bq_query(prod_desc_query, simple_query_params=query_params)
        model_adv_input = bq_query(adv_prod_desc_query, simple_query_params=query_params)

        sku_df, desc_list = prod_desc_manipulation(
            model_input, chunk_size, leaf_catg_cnt, top_skus_cnt, catg_cols,
            embedding_order=embedding_order,
        )
        print("Shape of the processed dataset: ", sku_df.shape)
        print(len(desc_list))

        vectors = create_word_embeddings(desc_list, chunk_size)
        print("Product level word embeddings list length: ", len(vectors))

        cat_emb_df = create_catg_embedding(vectors, df=sku_df)
        print("Category level word embedding dataset shape: ", cat_emb_df.shape)

        idx = build_annoy_index(n_trees, df=cat_emb_df)
        raw_output = calculate_catg_catg_similarity(n_neighbors, idx, df=cat_emb_df)
        print("Initial output shape: ", raw_output.shape)

        adv_catg = adv_prod_desc_manipulation(model_adv_input, chunk_size, leaf_catg_cnt, catg_cols)
        print("Number of advertisable categories: ", adv_catg.shape)

        print("\nResolving similarity threshold and model...")
        final_df, effective_threshold, model_id = resolve_threshold_and_model(
            raw_output, adv_catg, min_score_threshold, catg_batch_limit
        )
        print(f"Output shape after resolve: {final_df.shape}")

        final_df.to_csv(full_path, index=False, sep='\t', encoding='utf-8')
        s3_client = S3Client(access_key=access_key, secret_key=secret_key)
        s3_client.upload_s3_file(full_path, s3_output_path)
        print(f"Pipeline complete. Saved to {s3_output_path}")

    if run_validation:
        if not run_pipeline:
            full_path = f"/tmp/{output_file}"

        if 's3_client' not in dir():
            s3_client = S3Client(access_key=access_key, secret_key=secret_key)

        # Download validation cache from S3 (persists across Vertex runs)
        if pipeline_type == "opc":
            s3_cache_path = S3_OPC_VALIDATION_CACHE.format(client_id=client_id)
        else:
            s3_cache_path = S3_CATG_VALIDATION_CACHE.format(client_id=client_id)
        try:
            local_cache_path = s3_client.download_s3_file(s3_cache_path)
            os.makedirs("validation_cache", exist_ok=True)
            import shutil
            shutil.copy(local_cache_path, f"validation_cache/validation_cache_{client_id}.tsv")
            print(f"Validation cache downloaded from {s3_cache_path}")
        except Exception as e:
            print(f"No existing cache on S3 (first run): {e}")

        validate_with_llm(full_path, str(client_id), model_id=model_id,
                          system_instr=system_instr, system_instr_2=system_instr_2)

        # Upload all output files to S3
        upload_files = {
            "_validated.tsv": "Validated output",
            "_pass1.tsv": "Pass 1 output",
            "_llm_output.tsv": "LLM output",
        }
        for suffix, label in upload_files.items():
            local_path = full_path.replace(".tsv", suffix)
            if os.path.exists(local_path):
                s3_path = s3_output_path.replace(".tsv", suffix)
                s3_client.upload_s3_file(local_path, s3_path)
                print(f"{label} uploaded to {s3_path}")

        # Upload validation cache to S3
        cache_local = f"validation_cache/validation_cache_{client_id}.tsv"
        if os.path.exists(cache_local):
            s3_client.upload_s3_file(cache_local, s3_cache_path)
            print(f"Validation cache uploaded to {s3_cache_path}")

In [ ]:
if __name__ == "__main__":
    main(client_id=client_id)